In [ ]:
%cd "C:\Users\Public\Downloads\Uni Lecture\Data Preprocessing\Final's Project\Flight Delay\src"

C:\Users\Public\Downloads\Uni Lecture\Data Preprocessing\Final's Project\Flight Delay\src


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import json
from pathlib import Path
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from sklearn.preprocessing import PowerTransformer

from utils.data.dataset import Dataset, to_dataset

OUTPUT = Path('./plot')

In [4]:
RAW_PATH = Path('../data/engineered')

'''
--LOAD DATA--
'''
print('LOADING DATA...')
train_df = pd.read_csv(RAW_PATH / 'train_engineered.csv', low_memory=False)
print('DONE LOADING.')
print('--------------------------------')

LOADING DATA...
DONE LOADING.
--------------------------------


In [10]:
ds = to_dataset(train_df, 'arr_delay')

# 1. Dataset Overview

In [11]:
len(ds.x.columns), ds.x.columns

(45,
 Index(['month', 'day_of_month', 'day_of_week', 'op_unique_carrier', 'origin',
        'dest', 'crs_arr_time', 'crs_elapsed_time', 'distance', 'hour',
        'minute', 'temperature_2m', 'surface_pressure', 'relative_humidity_2m',
        'dew_point_2m', 'wind_speed_10m', 'wind_direction_10m', 'cloud_cover',
        'weather_Clear_Cloudy', 'weather_Rain', 'weather_Snow_Ice', 'season',
        'time_of_day', 'weather_severity_score', 'is_bad_weather',
        'rush_hour_x_weather', 'wind_exposure', 'wind_dir_sin', 'wind_dir_cos',
        'hour_sin', 'hour_cos', 'origin_congestion', 'dest_congestion',
        'weather_dist_risk', 'wind_route_risk', 'expected_speed',
        'is_peak_hour', 'is_weekend_rush', 'wind_risk', 'severe_ops_risk',
        'hub_pressure', 'long_haul_weather_risk', 'weekend_peak_risk',
        'adverse_weather_interaction', 'airport_operational_stress'],
       dtype='str'))

In [12]:
ds.x[['op_unique_carrier', 'origin', 'dest']].nunique()

op_unique_carrier     15
origin               319
dest                 319
dtype: int64

In [ ]:
datetime = pd.to_datetime(ds.x['fl_date'], format='mixed')
datetime.min(), datetime.max()

(Timestamp('2024-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

# 2. Delay Distribution

In [61]:
def get_hist_kde(series, bins=70):
    counts, bin_edges = np.histogram(series, bins=bins)
    kde_x = np.linspace(series.min(), series.max(), 300)
    kde_y = gaussian_kde(series)(kde_x)

    return {
        'hist': {'counts': counts.tolist(),
                 'bin_edges': bin_edges.tolist()},
        'kde': {'x': kde_x.tolist(),
                'y': kde_y.tolist()}
    }

In [ ]:
col = ds.y.dropna()

pt = PowerTransformer(method='yeo-johnson')
col_yj = pd.Series(pt.fit_transform(col.values.reshape(-1, 1)).flatten())

box_stats = {
    'min': float(col.min()),
    'q1': float(col.quantile(0.25)),
    'median': float(col.median()),
    'q3': float(col.quantile(0.75)),
    'max': float(col.max())
}

arr_delay_dist = {
    'original': {**get_hist_kde(col), 'box': box_stats},
    'yeo-johnson': get_hist_kde(col_yj)
}

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

sns.histplot(ds.y, bins=70, kde=True, ax=axes[0])
axes[0].set_xlabel('Delay Time (minutes)', fontsize=17)
axes[0].set_ylabel('Count', fontsize=17)

sns.histplot(ds.y, bins=70, kde=True, ax=axes[1])
axes[1].set_xlabel('Delay Time (minutes)', fontsize=17)
axes[1].set_ylabel('Count', fontsize=17)

In [60]:
with open(OUTPUT / 'arr_delay_dist.json', 'w') as f:
    json.dump(arr_delay_dist, f)

# 3. Missing Value Analysis

In [ ]:
missing = (train_df.isna().sum() / len(train_df)).sort_values(ascending=False)
missing = missing[missing > 0]
missing_dict = missing.round(6).to_dict()

In [ ]:
fig = plt.figure(figsize=(12, 10))

sns.barplot(x='x', y='column', data=missing.reset_index(name='column'),
            color='maroon')

In [ ]:
with open(OUTPUT / 'missing_values.json', 'w') as f:
    json.dump(missing_dict, f)

# 4. Categorical Grouping

In [69]:
def groupcat_counts(series, coverage):
    counts = series.value_counts(normalize=False)          # raw counts
    counts_pct = series.value_counts(normalize=True).sort_values(ascending=False)
    cum_pct = counts_pct.cumsum()
    n_keep = (cum_pct < coverage).sum() + 1
    n_keep = min(n_keep, len(counts_pct))
    keep_cat = counts_pct.index[:n_keep]

    pre = {str(k): int(v) for k, v in counts.items()}
    post = {str(k): int(counts[k]) for k in keep_cat}
    other_count = int(counts[~counts.index.isin(keep_cat)].sum())
    if other_count > 0:
        post["Other"] = other_count

    return {"pre": pre, "post": post}

In [70]:
COVERAGE = {
    'op_unique_carrier': 0.98,
    'origin': 0.999,
    'dest': 0.999
}

cat_output = {col: groupcat_counts(df[col], cov)
              for col, cov in COVERAGE.items()}

with open(OUTPUT / 'categorical_grouping.json', 'w') as f:
    json.dump(cat_output, f)

# 5. Correlation Heatmap

In [40]:
corr = ds_reduced.x.select_dtypes(include='number').corr().round(4)
corr_dict = corr.to_dict()

with open(OUTPUT / 'correlation.json', 'w') as f:
    json.dump(corr_dict, f)

# 6. Temporal Flight Frequency

In [43]:
temporal = {
    'day_of_week': ds_reduced.x.groupby('day_of_week').size().to_dict(),
    'day_of_month': ds_reduced.x.groupby('day_of_month').size().to_dict(),
    'monthly': ds_reduced.x.groupby('month').size().to_dict(),
    'season': ds_reduced.x.groupby('season').size().to_dict(),
}

with open(OUTPUT / 'temporal.json', 'w') as f:
    json.dump(temporal, f)

# 7. Box Plot on Hourly Delay Volatility

In [54]:
def box(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    filtered = series[(series >= q1 - 1.5 * iqr) & (series <= q3 + 1.5 * iqr)]
    
    return {
        'min': filtered.min(),
        'q1': q1,
        'median': filtered.median(),
        'q3': q3,
        'max': filtered.max()
    }

hourly = df_reduced.groupby('hour')['arr_delay'].apply(box).to_dict()
hourly = {str(k): v for k, v in hourly.items()}

with open(OUTPUT / 'hourly_delay.json', 'w') as f:
    json.dump(hourly, f)

In [65]:
df[['season', 'fl_date']]

,season,fl_date
0,0,2024-01-01
1,0,2024-01-01
2,0,2024-01-01
3,0,2024-01-01
4,0,2024-01-01
...,...,...
14080695,0,12/31/2025 12:00:00 AM
14080696,0,12/31/2025 12:00:00 AM
14080697,0,12/31/2025 12:00:00 AM
14080698,0,12/31/2025 12:00:00 AM
